# Baseline: Обучение Multi-Branch MLP на размеченных данных


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

from model import MultiBranchMLP
from data_module import DataModule
from lightning_module import BaseLightningModule

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


## 1. Загрузка данных


In [2]:
data_dir = '../data'

dm = DataModule(
    data_dir=data_dir,
    batch_size=128,
    num_workers=4
)

dm.setup()

print(f'Input dimension: {dm.input_dim}')
print(f'Number of classes: {dm.n_classes}')
print(f'Labeled train samples: {len(dm.train_labeled_dataset)}')
print(f'Test samples: {len(dm.test_dataset)}')


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000


## 2. Анализ дисбаланса классов и вычисление весов


In [3]:
train_labels = dm.train_labeled_dataset.y

unique_labels = np.unique(train_labels)
class_weights = compute_class_weight(
    'balanced',
    classes=unique_labels,
    y=train_labels
)

print(f'Class weights: {dict(zip(unique_labels, class_weights))}')

class_weights_tensor = torch.FloatTensor(class_weights)


Class weights: {0: 1.0738255033557047, 1: 0.963855421686747, 2: 1.0256410256410255, 3: 1.103448275862069, 4: 0.9523809523809523, 5: 0.935672514619883, 6: 0.9523809523809523, 7: 1.0596026490066226, 8: 0.9248554913294798, 9: 1.0457516339869282}


## 3. Создание модели


In [4]:
model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=256,
    output_dim=dm.n_classes,
    num_blocks=4,
    dropout=0.1,
    combine_mode='concat'
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')


Model parameters: 4,080,650


## 4. Создание Lightning модуля


In [5]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

lightning_model = BaseLightningModule(
    model=model,
    loss_fn=loss_fn,
    optimizer_type='adamw',
    learning_rate=1e-3,
    task_type='multiclass'
)


## 5. Обучение модели


In [6]:
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints',
    filename='best_model-{epoch:02d}-{val_accuracy:.4f}',
    monitor='val_accuracy',
    mode='max',
    save_top_k=1,
    save_last=True
)

trainer = Trainer(
    max_epochs=100,
    callbacks=[checkpoint_callback],
    enable_checkpointing=True,
    logger=True,
    enable_progress_bar=False,
    enable_model_summary=True,
    accelerator='auto',
    devices='auto'
)

trainer.fit(lightning_model, dm)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ MultiBranchMLP   │  4.1 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
│ 2 │ metrics │ ModuleDict       │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.1 M                                                                                                
Total estimated model params size (MB): 16                                                                         
Modules in train mode: 70                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Epoch 0: accuracy=0.1250, f1_macro=0.0485
Epoch 0: accuracy=0.1031, f1_macro=0.0304
Epoch 1: accuracy=0.1296, f1_macro=0.0703
Epoch 2: accuracy=0.1383, f1_macro=0.0901
Epoch 3: accuracy=0.1556, f1_macro=0.1213
Epoch 4: accuracy=0.1694, f1_macro=0.1494
Epoch 5: accuracy=0.1801, f1_macro=0.1650
Epoch 6: accuracy=0.1835, f1_macro=0.1653
Epoch 7: accuracy=0.1938, f1_macro=0.1744
Epoch 8: accuracy=0.2022, f1_macro=0.1872
Epoch 9: accuracy=0.2060, f1_macro=0.1945
Epoch 10: accuracy=0.2117, f1_macro=0.1994
Epoch 11: accuracy=0.2078, f1_macro=0.1941
Epoch 12: accuracy=0.2122, f1_macro=0.1962
Epoch 13: accuracy=0.2133, f1_macro=0.1998
Epoch 14: accuracy=0.2113, f1_macro=0.2019
Epoch 15: accuracy=0.2115, f1_macro=0.2036
Epoch 16: accuracy=0.2125, f1_macro=0.2059
Epoch 17: accuracy=0.2157, f1_macro=0.2106
Epoch 18: accuracy=0.2183, f1_macro=0.2147
Epoch 19: accuracy=0.2228, f1_macro=0.2192
Epoch 20: accuracy=0.2266, f1_macro=0.2233
Epoch 21: accuracy=0.2288, f1_macro=0.2256
Epoch 22: accuracy=0.2

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: accuracy=0.2784, f1_macro=0.2785


## 6. Оценка на тестовой выборке


In [7]:
best_model_path = checkpoint_callback.best_model_path
print(f'Loading best model from: {best_model_path}')

if best_model_path:
    best_model = BaseLightningModule.load_from_checkpoint(
        best_model_path,
        model=model,
        loss_fn=loss_fn,
        optimizer_type='adamw',
        learning_rate=1e-3,
        task_type='multiclass'
    )
else:
    best_model = lightning_model

test_results = trainer.test(best_model, dm)

print('\n=== Финальные результаты на тестовой выборке ===')
for key, value in test_results[0].items():
    print(f'{key}: {value:.4f}')


Loading best model from: /home/chistyakov-ivan/DL/dl2025/lesson7/homework/baseline/checkpoints/best_model-epoch=99-val_accuracy=0.2784-v4.ckpt


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000
Test results: accuracy=0.3530, f1_macro=0.3453


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.3529999852180481     │
│       test_f1_macro       │    0.3452816903591156     │
│         test_loss         │     11.48500919342041     │
└───────────────────────────┴───────────────────────────┘


=== Финальные результаты на тестовой выборке ===
test_loss: 11.4850
test_accuracy: 0.3530
test_f1_macro: 0.3453


## 7. После обучения делаем предсказания на неразмеченных данных

In [ ]:
# Загрузка неразмеченных данных
unlabeled_df = pd.read_csv('../data/train_unlabeled.csv')
X_unlabeled = unlabeled_df.values.astype(np.float32)

# Предсказание с помощью обученной baseline-модели
best_model.eval()
with torch.no_grad():
    logits = best_model(torch.from_numpy(X_unlabeled))
    probs = torch.softmax(logits, dim=1)
    max_probs, pseudo_labels = torch.max(probs, dim=1)

# Фильтрация по уверенности
confidence_threshold = 0.95
mask = max_probs >= confidence_threshold
X_pseudo = X_unlabeled[mask.cpu().numpy()]
y_pseudo = pseudo_labels[mask].cpu().numpy()

print(f"Сгенерировано {len(y_pseudo)} pseudo-labeled образцов.")

# DataModule с pseudo-labels
from data_module import DataModule

dm_ss = DataModule(
    data_dir='../data',
    batch_size=128,
    num_workers=4,
    pseudo_data=(X_pseudo, y_pseudo)
)

dm_ss.setup() 

model_ss = MultiBranchMLP(
    input_dim=dm_ss.input_dim,
    hidden_dim=256,
    output_dim=dm_ss.n_classes,
    num_blocks=4,
    dropout=0.1,
    combine_mode='concat'
)

y_orig = dm_ss.train_labeled_dataset.y
y_combined = np.concatenate([y_orig, y_pseudo])

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_combined),
    y=y_combined
)
class_weights_tensor = torch.FloatTensor(class_weights)

loss_fn_ss = nn.CrossEntropyLoss(weight=class_weights_tensor)

lightning_model_ss = BaseLightningModule(
    model=model_ss,
    loss_fn=loss_fn_ss,
    optimizer_type='adamw',
    learning_rate=1e-3,
    task_type='multiclass'
)

# Обучение
checkpoint_callback_ss = ModelCheckpoint(
    dirpath='checkpoints',
    filename='best_model_ss-{epoch:02d}-{val_accuracy:.4f}',
    monitor='val_accuracy',
    mode='max',
    save_top_k=1,
    save_last=True
)

trainer_ss = Trainer(
    max_epochs=100,
    callbacks=[checkpoint_callback_ss],
    accelerator='auto',
    devices='auto',
    enable_progress_bar=False
)

trainer_ss.fit(lightning_model_ss, dm_ss)
trainer_ss.test(lightning_model_ss, dm_ss)

Сгенерировано 10421 pseudo-labeled образцов.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


✅ Added 10421 pseudo-labeled samples.
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Total train samples (labeled + pseudo): 12021
Test samples: 4000


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


✅ Added 10421 pseudo-labeled samples.
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Total train samples (labeled + pseudo): 12021
Test samples: 4000


...

Epoch 0: accuracy=0.1016, f1_macro=0.0403
Epoch 0: accuracy=0.2770, f1_macro=0.2429
Epoch 1: accuracy=0.2720, f1_macro=0.2598
Epoch 2: accuracy=0.2807, f1_macro=0.2840
Epoch 3: accuracy=0.2843, f1_macro=0.2846
Epoch 4: accuracy=0.2939, f1_macro=0.2951
Epoch 5: accuracy=0.2915, f1_macro=0.2943
Epoch 6: accuracy=0.2961, f1_macro=0.2992
Epoch 7: accuracy=0.2955, f1_macro=0.2999
Epoch 8: accuracy=0.2967, f1_macro=0.3012
Epoch 9: accuracy=0.2980, f1_macro=0.3022
Epoch 10: accuracy=0.2967, f1_macro=0.3017
Epoch 11: accuracy=0.2993, f1_macro=0.3039
Epoch 12: accuracy=0.3005, f1_macro=0.3047
Epoch 13: accuracy=0.2992, f1_macro=0.3029
Epoch 14: accuracy=0.3010, f1_macro=0.3039
Epoch 15: accuracy=0.3021, f1_macro=0.3049
Epoch 16: accuracy=0.2958, f1_macro=0.2986
Epoch 17: accuracy=0.2918, f1_macro=0.2944
Epoch 18: accuracy=0.2906, f1_macro=0.2938
Epoch 19: accuracy=0.2910, f1_macro=0.2939
Epoch 20: accuracy=0.2919, f1_macro=0.2947
Epoch 21: accuracy=0.2927, f1_macro=0.2952
Epoch 22: accuracy=0.2

`Trainer.fit` stopped: `max_epochs=100` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


✅ Added 10421 pseudo-labeled samples.
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Total train samples (labeled + pseudo): 12021
Test samples: 4000
Test results: accuracy=0.3528, f1_macro=0.3505


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.3527500033378601     │
│       test_f1_macro       │    0.35053688287734985    │
│         test_loss         │    101.14622497558594     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 101.14622497558594,
  'test_accuracy': 0.3527500033378601,
  'test_f1_macro': 0.35053688287734985}]